## **REAL ESTATE APPRAISAL DASHBOARD COMPLETE ANALYSIS**

![image](https://www.adanirealty.com/-/media/Project/Realty/Blogs/What%20is%20real%20estate%20appraisal_%20Why%20does%20it%20matter_.png)

# Stage 1: Environment Setup & Data Ingestion
This stage mounts Google Drive, imports core data manipulation libraries, loads the raw housing datasets (`houses_data_v2.xlsx` and `egypt_real_estate_listings.xlsx`), and configures floating-point display formatting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

df = pd.read_excel('/content/drive/MyDrive/BANK PROJECT/houses_data_v2.xlsx')

df2 = pd.read_excel('/content/drive/MyDrive/BANK PROJECT/egypt_real_estate_listings.xlsx')

# Format display options for readable numbers
pd.set_option('display.float_format', '{:,.0f}'.format)

print("Datasets loaded successfully!")
print(f"df shape: {df.shape}")
print(f"df2 shape: {df2.shape}")

Datasets loaded successfully!
df shape: (23347, 10)
df2 shape: (19924, 11)


# Stage 2: Dataset 2 (Real Estate Listings) - Exploration, View Extraction & Cleaning
This stage explores Dataset 2 (`df2`), extracts property view categories using bilingual regular expressions (English and Arabic), standardizes room and bathroom attributes, imputes missing values using property-type medians, and cleans geographic features.

In [ ]:
# Preview initial datasets secound
display(df2.head())

,url,price,description,location,type,size,bedrooms,bathrooms,available_from,down_payment,Furnished
0,https://www.propertyfinder.eg/en/plp/buy/chale...,"8,000,000",OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...,"Swan Lake Gouna, Al Gouna, Hurghada, Red Sea",Chalet,732 sqft / 68 sqm,1+ Maid,1,2025-08-31,"1,200,000 EGP",No
1,https://www.propertyfinder.eg/en/plp/buy/villa...,"25,000,000","For sale, a villa with immediate delivery in C...","Karmell, New Zayed City, Sheikh Zayed City, Giza",Villa,"2,368 sqft / 220 sqm",4,4,2025-09-02,"2,100,000 EGP",No
2,https://www.propertyfinder.eg/en/plp/buy/chale...,"15,135,000","With a down payment of EGP 1,513,000, a fully ...","Azha North, Ras Al Hekma, North Coast",Chalet,"1,270 sqft / 118 sqm",2,2,2025-08-19,"1,513,000 EGP",No
3,https://www.propertyfinder.eg/en/plp/buy/apart...,"12,652,000",Own an apartment in New Cairo with a minimal d...,"Taj City, 5th Settlement Compounds, The 5th Se...",Apartment,"1,787 sqft / 166 sqm",3,2,2025-08-26,"1,260,000 EGP",No
4,https://www.propertyfinder.eg/en/plp/buy/villa...,"45,250,000",Project: Granville\nLocation: Fifth Settlement...,"Granville, New Capital City, Cairo",Villa,"4,306 sqft / 400 sqm",7,7,2025-09-02,"2,262,500 EGP",No


In [ ]:
# Inspect total non-null record counts across all columns in df2
df2.count()

,0
url,19924
price,19385
description,19846
location,19833
type,19847
size,19847
bedrooms,19780
bathrooms,19784
available_from,19261
down_payment,5445


In [ ]:
# Check initial missing value counts per column in df2
df2.isnull().sum()

,0
url,0
price,539
description,78
location,91
type,77
size,77
bedrooms,144
bathrooms,140
available_from,663
down_payment,14479


***there are some indicators for outliers and misleading data + a lot of missing***

***____________________________________________________________________***

# ***Cleaning***

---

***categories the veiw***

---



In [ ]:
#extracting all importan information from the describtion feature
import pandas as pd
import re

# 1.TWO REGEX PATTERNS (ENGLISH ANT ARABIC)

# ENGLISH VIEW
en_targets = (
    r'sea|seaside|beach|coast|coastal|ocean|shore|'
    r'lagoon|lagoons|lake|lakes|crystal\s+lagoon|crystal\s+lagoons|'
    r'pool|pools|swimming\s+pool|swimming\s+pools|'
    r'garden|gardens|land\s*scape|landscaping|greenery|green\s+spaces?|green\s+areas?|green\s+view|park|parks|central\s+park|'
    r'golf|golf\s+course|'
    r'river|nile|canal|canals|water|water\s+features?|water\s+bodies|waterfall|fountain|'
    r'city|skyline|downtown|landmark|'
    r'street|streets|road|roads|strip|main\s+street|wide\s+street|'
    r'courtyard|plaza|patio'
)

en_adj = r'(?:direct|partial|side|panoramic|unobstructed|full|prime|distinctive|exceptional|stunning|breathtaking|wide|open|clear|double|spectacular|amazing|front|scenic)'

# English Pattern
pattern_en = rf"""(?ix)
\b(
    # Target + view (e.g., "sea view", "green spaces view", "crystal lagoon views", "pool-view")
    (?:{en_targets})[\s\-]+views?\b

    # Adjective + view (e.g., "open view", "panoramic views", "unobstructed view", "double view")
  | {en_adj}\s+views?\b

    # View of / on / over / towards (e.g., "view of the sea", "views on pool", "view over lake")
  | views?\s*(?:of|on|over|onto|towards|across|to|at)?\s*(?:the\s+)?(?:{en_adj}\s+)?(?:{en_targets})\b

    # View with colon/dash (e.g., "view: sea", "views - pool", "views: open")
  | views?\s*[:\-\|]\s*(?:the\s+)?(?:{en_targets}|open|panoramic)\b

    # Overlooking / Facing (e.g., "overlooking green spaces", "facing the sea")
  | (?:overlooking|overlooks|looking\s+(?:over|onto|at|to)|facing|faces|fronting)\s*(?:the\s+)?(?:{en_adj}\s+)?(?:{en_targets})\b

    # Frontages (e.g., "seafront", "beachfront", "waterfront", "pool front")
  | \b(?:sea|beach|lagoon|lake|pool|golf|water)[\s\-]*(?:front|frontage)\b

    # Front / First row (e.g., "first row sea", "1st row beach", "first row lagoon")
  | \b(?:1st|first|2nd|second|front)\s+row\s+(?:on\s+)?(?:the\s+)?(?:sea|beach|lagoon|lake|water|pool|nile)\b

    # Direct on (e.g., "direct on sea", "directly on the lagoon", "direct on the nile")
  | \bdirect(?:ly)?\s+(?:on|to)\s+(?:the\s+)?(?:sea|beach|lagoon|lake|pool|garden|nile|water)\b
)"""

In [ ]:
#بلعربي بقا وي كمان لو في فرانكو

ar_targets = (
    # Sea
    r'بحر|شاط[ئى]|بلاج|ساحل|sea|beach|'
    # Lagoon / Lake
    r'لاجون|لاجونز|كريستال\s+لاجون|بحير[ةه]|بحيرات|lagoon|lagoons|lake|crystal\s+lagoon|'
    # Pool
    r' (?:ات)?\s*سباح[ةه]|بيسين|بول|pool|swimming\s+pool|'
    # Garden / Greenery
    r'جاردن|حديق[ةه]|حدائق|جنين[ةه]|لاند\s*سكيب|لاندسكيب|مساحات?\s*خضراء|مساح[ةه]\s*خضراء|بارك|سنترال\s*بارك|خضر[ةه]|greenery|land\s*scape|garden|central\s+park|park|'
    # Golf
    r'جولف|golf|'
    # Water / Nile
    r'نيل|نهر(?:\s+النيل)?|مجرى\s+مائي|مسطحات?\s*مائي[ةه]|نافورة|شلالات|water|nile|'
    # Street / Road
    r'شارع|شوارع|محور|طريق|street|road|'
    # City / Landmark
    r'مدين[ةه]|سيتي|معالم|city|landmark|'
    # Courtyard
    r'كورت\s*يارد|فناء|courtyard|'
    # Open View
    r'مفتوح|صريح|غير\s+مجروح[ةه]?|بانورام[اى]|رائع|مميز|خرافي|ساحر|open'
)

# Arabic Pattern
pattern_ar = rf"""(?ix)
(
    # فيو / الفيو متبوع بالهدف
    (?:فيو|الفيو)\s*(?:[:\-\|]|\s+)*(?:على|علي|ع|من|في)?\s*(?:ال)?(?:{ar_targets})

    # إطلالة / اطلالة / باطلالة على ...
  | (?:[إاأ]طلال[ةه]|باطلال[ةه]|بإطلال[ةه])\s*(?:[:\-\|]|\s+)*(?:مباشرة?\s*)?(?:على|علي|ع|من|تجاه)?\s*(?:ال)?(?:{ar_targets})

    # تطل / يطل / مطلة (تجنب مطلوب / المطلوب)
  | (?:تطل|يطل|مطل[ةه])\s*(?:مباشرة?\s*)?(?:على|علي|ع|تجاه)?\s*(?:ال)?(?:{ar_targets})

    # شايف / شايفة / يرى / ترى
  | (?:شايف[ةه]?|[تي]رى)\s*(?:ال)?(?:{ar_targets})

    # واجهة على ... أو واجهة مفتوحة / واجهة غير مجروحة / واجهة بحري
  | واجه[ةه]\s*(?:على|علي|ع)?\s*(?:ال)?(?:{ar_targets}|بحري)

    # صف أول / صف اول ع البحر / اللاجون
  | صف\s*(?:[أا]ول|اول|تاني|ثاني)\s*(?:على|علي|ع)?\s*(?:ال)?(?:بحر|شاط[ئى]|لاجون|بحير[ةه]|نيل|sea|beach|lagoon)

    # مباشر على البحر / اللاجون / الحديقة
  | (?:مباشر[ةه]?|دايركت)\s*(?:على|علي|ع)\s*(?:ال)?(?:بحر|شاط[ئى]|لاجون|بحير[ةه]|حمام\s*سباح[ةه]|نيل|sea|beach|lagoon)

    # بحري صريح / غير مجروحة
  | (?:بحري\s+صريح|غير\s+مجروح[ةه])
)"""

In [ ]:
# 2. EXTRACTION AND BILINGUAL CATEGORIZATION

extracted_en = df2['description'].str.extract(pattern_en, expand=False).str.strip().str.lower()
extracted_ar = df2['description'].str.extract(pattern_ar, expand=False).str.strip().str.lower()

# Combine: take English match  or fallback to Arabic match if English is null
extracted_views = extracted_en.combine_first(extracted_ar)

def group_views(val):
    if pd.isna(val) or not str(val).strip():
        return 'Standard / Not Specified'
    val = str(val).lower()

    if any(w in val for w in ['sea', 'seaside', 'beach', 'coast', 'ocean', 'بحر', 'شاطئ', 'شاطي', 'بلاج', 'ساحل']):
        return 'Sea View'

    elif any(w in val for w in ['lagoon', 'lake', 'لاجون', 'بحير']):
        return 'Lagoon / Lake View'

    elif any(w in val for w in ['pool', 'سباح', 'بيسين', 'بول']):
        return 'Pool View'

    elif any(w in val for w in ['garden', 'landscape', 'land scape', 'greenery', 'green', 'park', 'حديق', 'حدائق', 'جنين', 'لاند', 'خضر']):
        return 'Garden / Green View'

    elif any(w in val for w in ['golf', 'جولف']):
        return 'Golf View'

    elif any(w in val for w in ['water', 'river', 'nile', 'canal', 'fountain', 'نيل', 'نهر', 'مائي', 'مسطح']):
        return 'Water View'

    elif any(w in val for w in ['city', 'skyline', 'downtown', 'landmark', 'مدين', 'سيتي', 'معالم']):
        return 'City View'

    elif any(w in val for w in ['street', 'road', 'شارع', 'شوارع', 'محور', 'طريق']):
        return 'Street View'

    elif any(w in val for w in ['courtyard', 'plaza', 'patio', 'كورت', 'فناء']):
        return 'Courtyard View'

    elif any(w in val for w in ['open', 'panoramic', 'unobstructed', 'wide', 'distinctive', 'مفتوح', 'غير مجروح', 'صريح', 'بانورام', 'مميز', 'رائع']):
        return 'Open View'

    else:
        return 'Standard / Not Specified'

df2['view'] = extracted_views.apply(group_views)



 #3.Minimizes Unspecified

def fill_unspecified_fallback(row):
    current_view = row['view']
    if current_view != 'Standard / Not Specified':
        return current_view

    text = (str(row['description']) + ' ' + str(row.get('location', ''))).lower()

    # Coastal and Resort listings
    if any(k in text for k in ['sea', 'beach','بحر', 'شاطئ', 'شاطي']):
        return 'Sea View'

    # Lagoon and Lake
    if any(k in text for k in ['lagoon', 'lake', 'لاجون', 'بحير']):
        return 'Lagoon / Lake View'

    # Pool
    if any(k in text for k in ['pool', 'سباح', 'بيسين', 'بول']):
        return 'Pool View'

    # Golf
    if any(k in text for k in ['golf', 'جولف']):
        return 'Golf View'

    # Nile / Waterfront
    if any(k in text for k in ['nile', 'river', 'canal', 'نيل', 'نهر']):
        return 'Water View'

    # Garden / Landscape / Green spaces
    if any(k in text for k in ['garden', 'landscape', 'green', 'park', 'حديق', 'جنين', 'لاندسكيب', 'خضراء']):
        return 'Garden / Green View'

    # Street / Road Axis
    if any(k in text for k in ['street', 'road', 'شارع', 'محور', 'طريق']):
        return 'Street View'

    return 'Standard / Not Specified'

# Apply the fallback
df2['view'] = df2.apply(fill_unspecified_fallback, axis=1)



# Check distribution
print(df2['view'].value_counts())

view
Sea View                    4961
Garden / Green View         4164
Lagoon / Lake View          2863
Standard / Not Specified    2107
Pool View                   1824
Open View                   1522
Street View                 1511
Water View                   511
Golf View                    393
City View                     57
Courtyard View                11
Name: count, dtype: int64


**Confirm working Regex by checking 50 samples**

In [ ]:
# Get 2 random examples from Standard / Not Specified
df2.loc[
    df2['view'] == 'Standard / Not Specified',
    ['description', 'view']
].sample(50, random_state=42)

,description,view
9249,-Location : New cairo\n-Developer :Taj\n-Compo...,Standard / Not Specified
16351,Villa for Sale – 225 sqm – North Coast\nby sod...,Standard / Not Specified
3614,Apartment for sale in New Capital - Al MAQSAD ...,Standard / Not Specified
7750,An opportunity to own a chalet in the most lux...,Standard / Not Specified
7996,- فيلا للبيع 350م توين هاوس قرية لازوردي الكيل...,Standard / Not Specified
16583,Townhouse for sale in The Square Compound\nVer...,Standard / Not Specified
2082,شـقـة للـبيـع هاي سوبر لوكـس قـسـط.. وياتـلحـق...,Standard / Not Specified
17086,belle vie by emaar\n-standalone type Grace A-M...,Standard / Not Specified
17476,شقة للبيع في فيلا مميزة في الشيخ زايد، الحي ال...,Standard / Not Specified
18695,Apartment for sale in Courtyard Compound - SOD...,Standard / Not Specified


In [ ]:
# Check missing value counts in df2 after creating the 'view' feature
df2.isnull().sum()

,0
url,0
price,539
description,78
location,91
type,77
size,77
bedrooms,144
bathrooms,140
available_from,663
down_payment,14479


In [ ]:
# Select relevant features from df2
selected = [
    'price',
    'location',
    'type',
    'size',
    'bedrooms',
    'bathrooms',
    'view',
    'Furnished'
]
df2_sel = df2[selected].copy()

# Drop any unit that has no location as they are no longer for sale
df2_sel = df2_sel.dropna(subset=['location'])

# Convert price from string with commas to numeric float
df2_sel['price'] = df2_sel['price'].astype(str).str.replace(',', '').astype(float)

# Verify structure
df2_sel.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19833 entries, 0 to 19923
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   price      19385 non-null  float64
 1   location   19833 non-null  object 
 2   type       19833 non-null  object 
 3   size       19833 non-null  object 
 4   bedrooms   19766 non-null  object 
 5   bathrooms  19770 non-null  object 
 6   view       19833 non-null  object 
 7   Furnished  19833 non-null  object 
dtypes: float64(1), object(7)
memory usage: 1.4+ MB


In [ ]:
# Extract area in square meters (sqm) as numeric float
df2_sel['size_sqm'] = df2['size'].str.extract(r'([\d,]+)\s*sqm')[0].str.replace(',', '').astype(float)

# Drop redundant raw text size column and calculate price per sqm
df2_sel = df2_sel.drop(columns=['size'])
df2_sel['price_per_sqm'] = (df2_sel['price'] / df2_sel['size_sqm']).round()

df2_sel.head()

,price,location,type,bedrooms,bathrooms,view,Furnished,size_sqm,price_per_sqm
0,"8,000,000","Swan Lake Gouna, Al Gouna, Hurghada, Red Sea",Chalet,1+ Maid,1,Sea View,No,68,"117,647"
1,"25,000,000","Karmell, New Zayed City, Sheikh Zayed City, Giza",Villa,4,4,Garden / Green View,No,220,"113,636"
2,"15,135,000","Azha North, Ras Al Hekma, North Coast",Chalet,2,2,Standard / Not Specified,No,118,"128,263"
3,"12,652,000","Taj City, 5th Settlement Compounds, The 5th Se...",Apartment,3,2,Open View,No,166,"76,217"
4,"45,250,000","Granville, New Capital City, Cairo",Villa,7,7,Garden / Green View,No,400,"113,125"


In [ ]:
#MAKE THE FIX HERE
# Fix the bedrooms column

s = df2_sel["bedrooms"].astype(str).str.lower().str.strip()

# Extract the numeric bedroom count
df2_sel["bedrooms_cleaned"] = pd.to_numeric(
    s.str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce"
)

# Studio = 1 bedroom
df2_sel.loc[
    s.str.contains(r"\bstudio\b", na=False),
    "bedrooms_cleaned"
] = 1

# Maid = +1 bedroom
df2_sel.loc[
    s.str.contains(r"\bmaid\b", na=False),
    "bedrooms_cleaned"
] += 1

# Put the cleaned result back into the original column
df2_sel["bedrooms"] = df2_sel["bedrooms_cleaned"]

# Remove temporary column
df2_sel.drop(columns="bedrooms_cleaned", inplace=True)

# Check examples
display(df2_sel[["bedrooms"]].head(20))
df2_sel[['bedrooms', 'bathrooms']].head()

,bedrooms
0,2
1,4
2,2
3,3
4,7
5,3
6,4
7,6
8,3
9,4


,bedrooms,bathrooms
0,2,1
1,4,4
2,2,2
3,3,2
4,7,7


In [ ]:
# Check missing values in df2_sel following bedroom standardization
df2_sel.isnull().sum()

,0
price,448
location,0
type,0
bedrooms,67
bathrooms,63
view,0
Furnished,0
size_sqm,0
price_per_sqm,448


In [ ]:
                         #bathrooms sometimes has n+ instead of just n
df2_sel["bathrooms"] = (
    df2_sel["bathrooms"]
    .astype(str)
    .str.replace("+", "", regex=False)
    .replace("none", pd.NA)
)

df2_sel["bathrooms"] = pd.to_numeric(
    df2_sel["bathrooms"], errors="coerce"
)

In [ ]:

df2_sel["type"] = df2_sel["type"].replace({
    "Town House": "Townhouse",
    "Standalone Villa": "Stand Alone Villa"
})


#calculating the median for each type
medians = {}

for apartment_type in df2_sel["type"].dropna().unique():
    data = df2_sel[df2_sel["type"] == apartment_type]

    medians[apartment_type] = {
        "price": data["price"].median(),
        "bedrooms": data["bedrooms"].median(),
        "size_sqm": data["size_sqm"].median(),
        "bathrooms": data["bathrooms"].median()
    }

print(medians)


{'Chalet': {'price': 10250000.0, 'bedrooms': 3.0, 'size_sqm': 120.0, 'bathrooms': 2.0}, 'Villa': {'price': 28000000.0, 'bedrooms': 5.0, 'size_sqm': 325.0, 'bathrooms': 4.0}, 'Apartment': {'price': 6750000.0, 'bedrooms': 3.0, 'size_sqm': 155.0, 'bathrooms': 2.0}, 'Penthouse': {'price': 12500000.0, 'bedrooms': 4.0, 'size_sqm': 188.0, 'bathrooms': 3.0}, 'Twin House': {'price': 23000000.0, 'bedrooms': 4.0, 'size_sqm': 277.0, 'bathrooms': 4.0}, 'Duplex': {'price': 12300000.0, 'bedrooms': 4.0, 'size_sqm': 234.0, 'bathrooms': 3.0}, 'iVilla': {'price': 12496000.0, 'bedrooms': 4.0, 'size_sqm': 240.0, 'bathrooms': 3.0}, 'Townhouse': {'price': 18000000.0, 'bedrooms': 4.0, 'size_sqm': 225.0, 'bathrooms': 4.0}, 'Hotel Apartment': {'price': 13294000.0, 'bedrooms': 2.0, 'size_sqm': 92.5, 'bathrooms': 1.0}, 'Cabin': {'price': 13000000.0, 'bedrooms': 2.0, 'size_sqm': 56.0, 'bathrooms': 1.0}, 'Bulk Sale Unit': {'price': 6668000.0, 'bedrooms': nan, 'size_sqm': 87.0, 'bathrooms': 1.0}, 'Palace': {'price':

In [ ]:
# fix  missing values for price, bedrooms, size, and bathrooms using property-type medians
for apartment_type, values in medians.items():
    mask = df2_sel["type"] == apartment_type

    for column, median_value in values.items():
        df2_sel.loc[mask, column] = (
            df2_sel.loc[mask, column].fillna(median_value)
        )
  #re cal the price_per_sqm after filling missing prices
df2_sel['price_per_sqm'] = (df2_sel['price'] / df2_sel['size_sqm']).round()

Confirm ***Cleaning***

In [ ]:
# Verify missing value counts in df2_sel after median imputation
df2_sel.isnull().sum()

,0
price,0
location,0
type,0
bedrooms,67
bathrooms,63
view,0
Furnished,0
size_sqm,0
price_per_sqm,0


***that means that there are some types that does not even have one value for bedroom or bathroom***

In [ ]:
# Drop remaining records where bedrooms or bathrooms could not be filled
df2_sel = df2_sel.dropna(
    subset=["bedrooms", "bathrooms"]
).reset_index(drop=True)

In [ ]:
# Inspect DataFrame info and non-null counts after dropping un-imputable rows

df2_sel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19766 entries, 0 to 19765
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          19766 non-null  float64
 1   location       19766 non-null  object 
 2   type           19766 non-null  object 
 3   bedrooms       19766 non-null  float64
 4   bathrooms      19766 non-null  float64
 5   view           19766 non-null  object 
 6   Furnished      19766 non-null  object 
 7   size_sqm       19766 non-null  float64
 8   price_per_sqm  19766 non-null  float64
dtypes: float64(5), object(4)
memory usage: 1.4+ MB


In [ ]:
# Confirm that zero null values remain across all columns in df2_sel
df2_sel.isnull().sum()

,0
price,0
location,0
type,0
bedrooms,0
bathrooms,0
view,0
Furnished,0
size_sqm,0
price_per_sqm,0


In [ ]:
#the location is indexing in one big string
# Extract city and district/region from location string
df2_sel['city'] = df2_sel['location'].str.split(',').str[-1].str.strip()
df2_sel['region'] = df2_sel['location'].str.split(',').str[-2].str.strip()

# Drop raw text location column as city and region are now cleanly separated
df2_sel = df2_sel.drop(columns=['location'])

# Verify extracted geographic features
print("Top Cities:\n", df2_sel['city'].value_counts().head(5))
print("\nTop Regions:\n", df2_sel['region'].value_counts().head(5))

Top Cities:
 city
Cairo          7602
North Coast    5244
Giza           4221
Red Sea        1596
Suez            601
Name: count, dtype: int64

Top Regions:
 region
New Cairo City       5213
6 October City       2125
Sheikh Zayed City    1904
Ras Al Hekma         1793
Hurghada             1595
Name: count, dtype: int64


# Stage 3: Dataset 1 (Houses Data) - Schema Standardization & Imputation
This stage inspects Dataset 1 (`df`), renames and standardizes columns to match the unified schema, extracts numeric floor values, imputes missing values using property-type medians, and handles missing views.

### Dataset 1: Same process + Standardize Schema for Merging

### View column

In [ ]:
# Correct column spelling from 'veiw' to 'view'

df = df.rename(columns={
    'veiw': 'view'
})

In [ ]:
# Preview initial rows of Dataset 1
df.head()

,House_Type,Size,Bedrooms,Bathrooms,Floor,Furnished,Region,City,Price,view
0,Apartment,170,3,2,9,No,Zahraa Al Maadi,Cairo,1546400,Open View
1,Apartment,104,2,1,7,No,Nasr City,Cairo,950000,Water View
2,Apartment,160,3,2,1,No,Mostakbal City,Cairo,2100000,Street View
3,Apartment,160,3,3,Ground,No,New Cairo - El Tagamoa,Cairo,3994232,Pool View
4,Apartment,145,3,2,3,No,New Cairo - El Tagamoa,Cairo,370000,Garden / Green View


In [ ]:
# Inspect column data types and non-null counts in Dataset 1
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23347 entries, 0 to 23346
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   House_Type  23347 non-null  object
 1   Size        23347 non-null  int64 
 2   Bedrooms    23347 non-null  int64 
 3   Bathrooms   23347 non-null  int64 
 4   Floor       23347 non-null  object
 5   Furnished   23347 non-null  object
 6   Region      23347 non-null  object
 7   City        23347 non-null  object
 8   Price       23347 non-null  int64 
 9   view        22720 non-null  object
dtypes: int64(4), object(6)
memory usage: 1.8+ MB


In [ ]:
# Check non-null counts across features in Dataset 1
df.count()

,0
House_Type,23347
Size,23347
Bedrooms,23347
Bathrooms,23347
Floor,23347
Furnished,23347
Region,23347
City,23347
Price,23347
view,22720


In [ ]:
# Check initial missing value counts per column in Dataset 1
df.isnull().sum()

,0
House_Type,0
Size,0
Bedrooms,0
Bathrooms,0
Floor,0
Furnished,0
Region,0
City,0
Price,0
view,627


# **Cleaning**

In [ ]:
# Rename columns in df to match df2_sel schema
df = df.rename(columns={
    'House_Type': 'type',
    'Size': 'size_sqm',
    'Bedrooms': 'bedrooms',
    'Bathrooms': 'bathrooms',
    'Floor': 'floor',
    'Furnished': 'Furnished',
    'Region': 'region',
    'City': 'city',
    'Price': 'price'
})

In [ ]:
# Convert 'Ground' to 0 and extract integer floor count (Int64 allows missing values)
df['floor'] = (
    df['floor']
    .astype(str)
    .str.lower()
    .str.replace('ground', '0')
    .str.extract(r'(\d+)')[0]
    .astype('Int64')
)

# Calculate price per square meter for df
df['price_per_sqm'] = (df['price'] / df['size_sqm']).round()

**Fill missing**

In [ ]:
# Inspect property type distribution in Dataset 1
df["type"].value_counts()

,count
type,
Apartment,19150
Duplex,1255
Town House,888
Stand Alone Villa,570
Penthouse,496
Studio,448
Twin House,304
Chalet,210
Room,22


In [ ]:
# Calculate the median for each house type
# and use it to fill missing values

df["type"] = df["type"].replace({
    "Town House": "Townhouse",
    "Standalone Villa": "Stand Alone Villa"
})

for house_type in df["type"].dropna().unique():

    mask = df["type"] == house_type

    # Calculate medians for this house type
    price_median = df.loc[mask, "price"].median()
    bedrooms_median = df.loc[mask, "bedrooms"].median()
    size_median = df.loc[mask, "size_sqm"].median()
    bathrooms_median = df.loc[mask, "bathrooms"].median()
    floor_median = df.loc[mask, "floor"].median()

    # Fill missing values using the median of the same house type
    df.loc[mask, "price"] = df.loc[mask, "price"].fillna(price_median)
    df.loc[mask, "bedrooms"] = df.loc[mask, "bedrooms"].fillna(bedrooms_median)
    df.loc[mask, "size_sqm"] = df.loc[mask, "size_sqm"].fillna(size_median)
    df.loc[mask, "bathrooms"] = df.loc[mask, "bathrooms"].fillna(bathrooms_median)
    df.loc[mask, "floor"] = df.loc[mask, "floor"].fillna(floor_median)

In [ ]:
#new price sqm
df['price_per_sqm'] = (df['price'] / df['size_sqm']).round()

In [ ]:
# Check remaining missing value counts in Dataset 1 after imputation
df.isnull().sum()

,0
type,0
size_sqm,0
bedrooms,0
bathrooms,0
floor,0
Furnished,0
region,0
city,0
price,0
view,627


In [ ]:
# Drop records with missing view values in Dataset 1
df = df.dropna(
    subset=["view"]
).reset_index(drop=True)

# Stage 4: Dataset Merging & Post-Merge Harmonization
This stage combines `df` and `df2_sel` into a single master DataFrame (`combined_df`), imputes unshared attributes (`view`, `Furnished`, `floor`) with standardized defaults, and exports an interim merged CSV file.

In [ ]:
# Combine both datasets into one master dataframe
combined_df = pd.concat([df, df2_sel], ignore_index=True)

# Check combined shape
print("Combined Shape:", combined_df.shape)

Combined Shape: (42486, 11)


In [ ]:
# Check missing value counts in the newly merged master DataFrame
combined_df.isnull().sum()

,0
type,0
size_sqm,0
bedrooms,0
bathrooms,0
floor,19766
Furnished,0
region,0
city,0
price,0
view,0


In [ ]:
# 1. Unshared features: fill missing values with clean, standardized defaults
combined_df['view'] = combined_df['view'].fillna('Standard / Not Specified')
combined_df['Furnished'] = combined_df['Furnished'].fillna('Unknown')
# Note: 'floor' is kept as nullable integer (Int64), so missing floors appear as <NA> instead of text

# 2. Check null values and dtypes across all columns
print("Column data types and nulls:")
print(combined_df.dtypes)
print("\nNull counts per column:")
print(combined_df.isna().sum())

Column data types and nulls:
type              object
size_sqm         float64
bedrooms         float64
bathrooms        float64
floor              Int64
Furnished         object
region            object
city              object
price            float64
view              object
price_per_sqm    float64
dtype: object

Null counts per column:
type                 0
size_sqm             0
bedrooms             0
bathrooms            0
floor            19766
Furnished            0
region               0
city                 0
price                0
view                 0
price_per_sqm        0
dtype: int64


In [ ]:
# 1. Replace 'Ground' with 0 and '10+' with 10
combined_df['floor'] = (
    combined_df['floor']
    .astype(str)
    .str.lower()
    .str.replace('ground', '0')
    .str.replace('10+','10', regex=False)
)

# 2. Convert text numbers to numeric (turns 'highest' or 'unknown' into NaN)
combined_df['floor'] = pd.to_numeric(combined_df['floor'], errors='coerce')

# 3. Convert to nullable integer 'Int64' (capital 'I' allows integers with missing values)
combined_df['floor'] = combined_df['floor'].astype('Int64')

# 4. Check the results
combined_df['floor'].value_counts(dropna=False).head(10)

,count
floor,
<NA>,19766
2,7215
3,4966
1,3378
0,2573
4,2003
5,1026
10,528
6,380


In [ ]:
# Verify remaining null counts across all columns in combined_df
combined_df.isna().sum()


,0
type,0
size_sqm,0
bedrooms,0
bathrooms,0
floor,19766
Furnished,0
region,0
city,0
price,0
view,0


In [ ]:
# Inspect structural summary and data types of combined_df
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42486 entries, 0 to 42485
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           42486 non-null  object 
 1   size_sqm       42486 non-null  float64
 2   bedrooms       42486 non-null  float64
 3   bathrooms      42486 non-null  float64
 4   floor          22720 non-null  Int64  
 5   Furnished      42486 non-null  object 
 6   region         42486 non-null  object 
 7   city           42486 non-null  object 
 8   price          42486 non-null  float64
 9   view           42486 non-null  object 
 10  price_per_sqm  42486 non-null  float64
dtypes: Int64(1), float64(5), object(5)
memory usage: 3.6+ MB


In [ ]:
# Export to CSV
combined_df.to_csv('combined.csv', index=False, encoding='utf-8-sig')

print("Successfully exported! Total rows:", len(combined_df))

Successfully exported! Total rows: 42486


# Stage 5: Exploratory Data Analysis & Outlier Handling
This stage explores property sizes and prices, removes unit types outside the bank's operational scope and low-frequency cities, applies property-type size boundaries with median replacement, and determines price-per-sqm outlier limits using the IQR method.

## Descriptive Statistics

In [ ]:
display(combined_df.describe())
#display(combined_df.groupby('type')['price_per_sqm'].describe())
#display(combined_df.groupby('type')['price'].skew())

,size_sqm,bedrooms,bathrooms,floor,price,price_per_sqm
count,"42,486","42,486","42,486","22,720","42,486","42,486"
mean,565,3,3,3,"8,398,419","40,016"
std,"76,711",1,1,2,"17,663,571","107,582"
min,1,1,1,0,"2,000",1
25%,128,2,2,1,"737,662","4,667"
50%,165,3,2,2,"3,300,000","20,000"
75%,220,4,3,3,"9,800,000","60,000"
max,"15,811,692",8,7,10,"840,000,000","19,500,000"


In [ ]:
# Calculate mean listing price by property type
combined_df.groupby("type")["price"].mean()

,price
type,
Apartment,"3,358,535"
Cabin,"14,677,658"
Chalet,"11,144,819"
Duplex,"5,841,294"
Full Floor,"8,606,250"
Hotel Apartment,"16,440,802"
Palace,"153,208,167"
Penthouse,"8,865,056"
Roof,"5,231,667"


In [ ]:
# getting the 25 largest apartments by size

top_25_size = combined_df.nlargest(50, "size_sqm")

top_25_size

,type,size_sqm,bedrooms,bathrooms,floor,Furnished,region,city,price,view,price_per_sqm
27127,Apartment,"15,811,692",3,3,<NA>,No,Mostakbal City - Future City,Cairo,"11,692,311",Standard / Not Specified,1
38158,Chalet,"96,106",2,2,<NA>,No,Ras Al Hekma,North Coast,"6,550,000",Sea View,68
30889,Apartment,"6,779",2,1,<NA>,No,Hay Sharq,Alexandria,"3,500,000",Open View,516
35338,Villa,"3,800",7,7,<NA>,No,New Cairo City,Cairo,"650,000,000",Pool View,"171,053"
32073,Villa,"3,300",8,7,<NA>,No,New Cairo City,Cairo,"28,000,000",Golf View,"8,485"
36171,Villa,"3,050",8,7,<NA>,No,New Cairo City,Cairo,"28,000,000",Standard / Not Specified,"9,180"
39768,Whole Building,"3,000",7,7,<NA>,No,Hay El Haram,Giza,"50,000,000",Pool View,"16,667"
41059,Villa,"3,000",8,7,<NA>,Yes,New Cairo City,Cairo,"100,000,000",Sea View,"33,333"
32555,Villa,"2,801",8,7,<NA>,No,New Cairo City,Cairo,"375,000,000",Pool View,"133,881"
40676,Palace,"2,450",8,7,<NA>,No,New Cairo City,Cairo,"300,000,000",Pool View,"122,449"


# A 1 METER SIZE HOW IS THAT ?

# **WHY?**

WE HAVE DIFFRENT TYPES OF EVERY THING VILLAS AND DUPLEX ETC..

SO Instead of gather them all and **describe**

*lets split them into*  **economic segments**

In [ ]:
# Analyze size percentiles by property type to detect unrealistic boundaries
import pandas as pd

size_stats = (
    df2_sel
    .groupby("type")["size_sqm"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

size_stats

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
type,,,,,,,,,,,,,,
Apartment,"8,355","2,049","172,982",29,48,70,88,125,155,185,220,245,330,"15,811,692"
Cabin,37,70,38,35,35,35,40,45,56,70,122,153,186,200
Chalet,"4,037",153,"1,512",1,41,64,72,98,120,147,190,226,370,"96,106"
Duplex,622,243,88,70,100,140,151,188,234,279,345,389,500,"1,100"
Full Floor,4,268,72,180,182,189,198,225,282,325,328,329,330,330
Hotel Apartment,104,106,56,33,34,42,46,64,92,145,170,222,246,305
Palace,23,"1,010",482,300,322,401,434,920,950,"1,025","1,680","1,871","2,324","2,450"
Penthouse,569,204,76,50,84,110,125,157,188,245,300,346,450,660
Roof,6,151,66,60,63,75,90,123,146,169,216,238,256,260


In [ ]:
# the bank does not work with this type of units why keeping i t ?
combined_df["type"].value_counts()

,count
type,
Apartment,27004
Chalet,4241
Villa,3562
Townhouse,2189
Duplex,1844
Twin House,1129
Penthouse,1047
Stand Alone Villa,558
Studio,434


# **detect and Fix outliers**

1.   determine the scope

1.   figure out the outlier boundaries
3.   fixing




**determine the scope**

In [ ]:
# Remove property types outside the bank's operational scope
types_to_remove = ["Room", "Whole Building", "Full Floor","Palace","Cabin","Hotel Apartment"]

combined_df = combined_df[
    ~combined_df["type"].isin(types_to_remove)
].copy()

In [ ]:
#detecting not usefull cities
city_counts = combined_df["city"].value_counts()


print(city_counts.head(35))


city
Cairo             23678
Giza               8365
North Coast        5159
Alexandria         2790
Red Sea            1578
Suez                597
Qalyubia             57
South Sainai         28
Al Daqahlya          11
Matrouh               6
Sharqia               4
Asyut                 4
Demyat                2
Aswan                 1
Luxor                 1
Kafr El Sheikh        1
Name: count, dtype: int64


In [ ]:
#deleting unusefull cities
#we do not have enough data to cover all of egypt
cities_to_delete = [
    "Qalyubia",
    "South Sainai",
    "Al Daqahlya",
    "Matrouh",
    "Sharqia",
    "Asyut",
    "Demyat",
    "Aswan",
    "Luxor",
    "Kafr El Sheikh"
]

combined_df = combined_df[
    ~combined_df["city"].isin(cities_to_delete)
].reset_index(drop=True)

In [ ]:
size_limits = {
    "Apartment": (48, 330),
    "Chalet": (41, 370),
    "Duplex": (100, 500),
    "Penthouse": (84, 450),
    "Roof": (63, 256),
    "Townhouse": (130, 500),
    "Twin House": (133, 600),
    "Villa": (122, 1576),
    "iVilla": (165, 324),
    "Studio": (40, 120)
}

# Calculate median using only values inside the valid limits
medians = {}

for property_type, (lower, upper) in size_limits.items():

    mask = (
        combined_df["type"].eq(property_type) &
        combined_df["size_sqm"].between(lower, upper)
    )

    medians[property_type] = combined_df.loc[mask, "size_sqm"].median()


# Replace values outside the limits with the median of their type
for property_type, (lower, upper) in size_limits.items():

    type_mask = combined_df["type"].eq(property_type)

    invalid_mask = (
        type_mask &
        ~combined_df["size_sqm"].between(lower, upper)
    )

    combined_df.loc[invalid_mask, "size_sqm"] = medians[property_type]
    combined_df["price_per_sqm"] = (
    combined_df["price"] / combined_df["size_sqm"]
).round(2)

**Checking if limits were fully applied**

In [ ]:
# Confirm that no listings violate the defined size limits
for property_type, (lower, upper) in size_limits.items():

    bad = combined_df[
        (combined_df["type"] == property_type) &
        (~combined_df["size_sqm"].between(lower, upper))
    ]

    print(property_type, len(bad))

Apartment 0
Chalet 0
Duplex 0
Penthouse 0
Roof 0
Townhouse 0
Twin House 0
Villa 0
iVilla 0
Studio 0


In [ ]:
#IQR AND BOUNDS (price/sqm)
q1 = 10909
q3 = 67347
IQR = q3 - q1
print(f'SALE UPPER BOUND: {q3 + 1.5*IQR}')
print(f'SALE LOWER BOUND: {q1 - 1.5*IQR}')

SALE UPPER BOUND: 152004.0
SALE LOWER BOUND: -73748.0


In [ ]:
#there is a big part of data is considered as outlier
high_price_sqm = combined_df[
    combined_df["price_per_sqm"] > 152_004
]

high_price_sqm

,type,size_sqm,bedrooms,bathrooms,floor,Furnished,region,city,price,view,price_per_sqm
22703,Chalet,126,3,2,<NA>,No,Al Alamein,North Coast,"19,760,000",Lagoon / Lake View,"156,825"
22715,Chalet,150,4,4,<NA>,No,Hurghada,Red Sea,"29,310,000",Sea View,"195,400"
22716,Chalet,72,2,1,<NA>,No,Qesm Ad Dabaah,North Coast,"14,127,000",Sea View,"196,208"
22728,Chalet,102,3,2,<NA>,No,Sidi Heneish,North Coast,"18,000,000",Lagoon / Lake View,"176,471"
22733,Villa,403,8,5,<NA>,No,Sheikh Zayed City,Giza,"71,500,000",Garden / Green View,"177,419"
...,...,...,...,...,...,...,...,...,...,...,...
42021,Villa,645,4,5,<NA>,No,New Cairo City,Cairo,"110,000,000",Standard / Not Specified,"170,543"
42026,Chalet,140,3,2,<NA>,No,Sidi Abdel Rahman,North Coast,"27,000,000",Sea View,"192,857"
42051,Chalet,140,2,2,<NA>,No,Sidi Abdel Rahman,North Coast,"25,000,000",Open View,"178,571"
42074,Villa,600,6,5,<NA>,No,Sidi Abdel Rahman,North Coast,"95,000,000",Lagoon / Lake View,"158,333"


# Stage 6: Economic Segmentation (Classes A–D) & Price Anomaly Filtering
This stage segments properties into 4 economic classes (A, B, C, D) using price-per-sqm quartiles, validates segment boundaries, detects and filters unrealistic low prices (< 400,000 EGP), checks upper-range listings, and exports the final cleaned dataset.

**based on the boundires:**
   we determine that spliting them into 4 segments is the most convienent

In [ ]:
#1. Divide the apartments into 4 classes based on price per square meter
combined_df["class"] = pd.qcut(
    combined_df["price_per_sqm"],
    q=4,
    labels=["D", "C", "B", "A"]
)

# 2. Display the number of apartments in each class
print("Number of apartments in each class:")
print(combined_df["class"].value_counts().sort_index())


# 3. Display descriptive statistics for each class
for label in ["A", "B", "C", "D"]:

    print(f"\n{'=' * 50}")
    print(f"CLASS {label}")
    print(f"{'=' * 50}")

    class_data = combined_df[combined_df["class"] == label]

    print(class_data.describe().round(2))

Number of apartments in each class:
class
D    10544
C    10540
B    10542
A    10541
Name: count, dtype: int64

CLASS A
       size_sqm  bedrooms  bathrooms  floor       price  price_per_sqm
count    10,541    10,541     10,541      3      10,541         10,541
mean        215         4          3      4  24,006,085        108,242
std         151         1          1      5  28,387,344         61,746
min          44         1          1      1   2,900,000         59,655
25%         125         3          2      2  11,275,000         74,074
50%         175         4          3      2  16,800,000         92,683
75%         255         4          4      6  26,500,000        122,607
max       1,560         8          7     10 840,000,000      2,000,000

CLASS B
       size_sqm  bedrooms  bathrooms  floor      price  price_per_sqm
count    10,542    10,542     10,542  2,596     10,542         10,542
mean        185         3          3      2  6,846,792         36,168
std         115      

**testing if the 4 segments actully works and not just forced**

In [ ]:
# Evaluate the minimum and maximum prices within each economic class
min_price_by_class = combined_df.groupby("class")["price"].min()
max_price_by_class = combined_df.groupby("class")["price"].max()

print(min_price_by_class)
print(max_price_by_class)

class
D       2,000
C     200,000
B     989,000
A   2,900,000
Name: price, dtype: float64
class
D     1,750,000
C    13,590,000
B    75,000,000
A   840,000,000
Name: price, dtype: float64


/tmp/ipykernel_2185/1383816856.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  min_price_by_class = combined_df.groupby("class")["price"].min()
/tmp/ipykernel_2185/1383816856.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_price_by_class = combined_df.groupby("class")["price"].max()


the D class seems to have a lot of outliers let's check it

In [ ]:
# Bin listings across price ranges to examine low-price distribution in Class D
bins = [0, 50_0000, 1_000_000, 2_000_000, 5_000_000, float("inf")]
labels = ["0-500K", "500K-1M", "1M-2M", "2M-5M", "5M+"]

price_ranges = pd.cut(
    combined_df["price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print(price_ranges.value_counts().sort_index())

price
0-500K      9578
500K-1M     2191
1M-2M       4614
2M-5M       9776
5M+        16008
Name: count, dtype: int64


In [ ]:
# Sample 10 listings priced under 400,000 EGP for validation
sample_10 = combined_df[
    combined_df["price"] <= 400_000
].sample(10, random_state=42)

sample_10

,type,size_sqm,bedrooms,bathrooms,floor,Furnished,region,city,price,view,price_per_sqm,class
448,Studio,48,1,1,3,Yes,New Cairo - El Tagamoa,Cairo,"4,000",Garden / Green View,83,D
2032,Apartment,100,2,1,1,Yes,Miami,Alexandria,"3,500",Open View,35,D
22299,Apartment,150,3,1,8,No,Azarita,Alexandria,"14,000",Open View,93,D
5496,Apartment,120,3,1,5,No,Nakheel,Alexandria,"310,000",Open View,"2,583",D
372,Apartment,150,3,2,3,No,New Cairo - El Tagamoa,Cairo,"390,000",Open View,"2,600",D
8641,Apartment,126,2,2,7,Yes,New Capital City,Cairo,"251,600",Water View,"1,997",D
17757,Apartment,115,3,1,2,No,New Cairo - El Tagamoa,Cairo,"155,000",Garden / Green View,"1,348",D
21985,Apartment,220,3,2,5,No,Dokki,Giza,"22,000",Street View,100,D
12015,Studio,105,1,2,3,No,Sheikh Zayed,Giza,"14,000",Open View,133,D
17118,Apartment,168,3,2,2,No,New Capital City,Cairo,"235,000",Open View,"1,399",D


In [ ]:
#AFTER CHECKING THE URL THE APPARTMENT THAT HAS A PRICE LOWER THAN 400K
# MOSTLY IS NOT REAL OR OLD
combined_df = combined_df[combined_df["price"] >= 400000]

In [ ]:
#highest 10 prices by price / area
combined_df.nlargest(10, 'price_per_sqm')

,type,size_sqm,bedrooms,bathrooms,floor,Furnished,region,city,price,view,price_per_sqm,class
35081,Villa,325,7,7,<NA>,No,New Cairo City,Cairo,"650,000,000",Pool View,"2,000,000",A
35848,Villa,325,8,7,<NA>,No,New Cairo City,Cairo,"400,000,000",Lagoon / Lake View,"1,230,769",A
32318,Villa,325,8,7,<NA>,No,New Cairo City,Cairo,"375,000,000",Pool View,"1,153,846",A
35006,Villa,325,8,7,<NA>,No,New Cairo City,Cairo,"350,000,000",Pool View,"1,076,923",A
31224,Villa,325,8,7,<NA>,No,New Cairo City,Cairo,"300,000,000",Pool View,"923,077",A
23083,Villa,345,7,7,<NA>,No,Qesm Marsa Matrouh,North Coast,"280,000,000",Water View,"811,594",A
34218,Villa,325,8,7,<NA>,Yes,New Cairo City,Cairo,"250,000,000",Street View,"769,231",A
35937,Villa,325,8,7,<NA>,Yes,New Cairo City,Cairo,"250,000,000",Golf View,"769,231",A
37145,Villa,"1,100",6,6,<NA>,No,Sidi Abdel Rahman,North Coast,"840,000,000",Sea View,"763,636",A
40828,Villa,610,5,5,<NA>,No,Sidi Abdel Rahman,North Coast,"453,000,000",Sea View,"742,623",A


In [ ]:
#checking the Url for appartment above 500k
#result appears to be normal and real
sample_10 = (
    combined_df[combined_df["price"] > 500_000]
    .sort_values("price")
    .head(10)
)

sample_10

,type,size_sqm,bedrooms,bathrooms,floor,Furnished,region,city,price,view,price_per_sqm,class
19629,Apartment,120,2,1,3,Yes,Faisal,Giza,"500,030",Open View,"4,167",D
17295,Apartment,129,3,3,2,No,New Capital City,Cairo,"501,100",Street View,"3,884",D
4893,Penthouse,161,4,4,3,No,New Cairo - El Tagamoa,Cairo,"502,000",Street View,"3,118",D
21208,Townhouse,205,3,3,2,No,6th of October,Giza,"503,000",Pool View,"2,454",D
15812,Duplex,337,5,4,3,No,Madinaty,Cairo,"504,000",Garden / Green View,"1,496",D
18299,Townhouse,165,3,3,4,No,Nasr City,Cairo,"504,000",Open View,"3,055",D
1386,Duplex,225,4,3,1,No,New Cairo - El Tagamoa,Cairo,"505,000",Garden / Green View,"2,244",D
8431,Apartment,244,3,2,1,No,New Cairo - El Tagamoa,Cairo,"505,000",Pool View,"2,070",D
5242,Apartment,202,3,2,6,No,Haram,Giza,"505,000",Garden - Green View,"2,500",D
7787,Penthouse,161,3,3,3,No,New Cairo - El Tagamoa,Cairo,"505,000",Pool View,"3,137",D


In [ ]:
# Export the final cleaned dataset to CSV and download
from google.colab import files

combined_df.to_csv("combined_df.csv", index=False)
files.download("combined_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Stage 7: Inferential Statistical Analysis across Economic Segments
This stage recalculates the 4 equal-sized economic segments post-filtering, prepares class subsets, and executes 10 comprehensive inferential statistical analyses (confidence intervals, bootstrap medians, t-tests, ANOVA, Kruskal-Wallis, Spearman correlation, and Chi-square independence tests).

# **Inferential-Statistics**





In [ ]:
# Import libraries for inferential statistics
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import (
    ttest_ind,
    mannwhitneyu,
    f_oneway,
    kruskal,
    levene,
    chi2_contingency,
    pearsonr,
    spearmanr
)

In [ ]:
# Divide apartments into 4 equal-sized segments
# based on price per square meter

combined_df["class"] = pd.qcut(
    combined_df["price_per_sqm"],
    q=4,
    labels=["D", "C", "B", "A"]
)

# Create separate DataFrames for each class

class_A = combined_df[combined_df["class"] == "A"].copy()
class_B = combined_df[combined_df["class"] == "B"].copy()
class_C = combined_df[combined_df["class"] == "C"].copy()
class_D = combined_df[combined_df["class"] == "D"].copy()

# Store them together so the 12 cells can loop through them

class_dfs = {
    "A": class_A,
    "B": class_B,
    "C": class_C,
    "D": class_D
}

# Check the number of apartments in each class

print("Number of apartments in each class:")
print(combined_df["class"].value_counts().sort_index(ascending=False))

Number of apartments in each class:
class
A    8322
B    8369
C    8351
D    8359
Name: count, dtype: int64


In [ ]:
# Test 1: Compute 95% Confidence Intervals for Mean Price across economic classes
results = []

for label, data in class_dfs.items():

    price_data = data["price"].dropna()

    mean_price = price_data.mean()
    std = price_data.std()
    n = len(price_data)

    standard_error = std / np.sqrt(n)

    lower, upper = stats.t.interval(
        confidence=0.95,
        df=n - 1,
        loc=mean_price,
        scale=standard_error
    )

    results.append({
        "Class": label,
        "Mean Price": mean_price,
        "95% CI Lower": lower,
        "95% CI Upper": upper
    })

results = pd.DataFrame(results)

display(results.round(2))

,Class,Mean Price,95% CI Lower,95% CI Upper
0,A,"26,659,338","25,992,359","27,326,316"
1,B,"10,140,377","9,976,251","10,304,502"
2,C,"3,622,322","3,580,961","3,663,682"
3,D,"1,479,529","1,460,820","1,498,239"


In [ ]:
# Test 2: Compute 95% Confidence Intervals for Mean Price per m² across economic classes
results = []

for label, data in class_dfs.items():

    price_sqm = data["price_per_sqm"].dropna()

    mean_price_sqm = price_sqm.mean()
    std = price_sqm.std()
    n = len(price_sqm)

    standard_error = std / np.sqrt(n)

    lower, upper = stats.t.interval(
        confidence=0.95,
        df=n - 1,
        loc=mean_price_sqm,
        scale=standard_error
    )

    results.append({
        "Class": label,
        "Mean Price/m²": mean_price_sqm,
        "95% CI Lower": lower,
        "95% CI Upper": upper
    })

results = pd.DataFrame(results)

display(results.round(2))

,Class,Mean Price/m²,95% CI Lower,95% CI Upper
0,A,"119,643","118,249","121,037"
1,B,"50,066","49,812","50,320"
2,C,"20,794","20,693","20,895"
3,D,"7,829","7,759","7,900"


In [ ]:
# Test 3: Compute 95% Bootstrap Confidence Intervals (5,000 resamples) for Median Price
np.random.seed(42)

results = []

for label, data in class_dfs.items():

    price_data = data["price"].dropna().values

    bootstrap_medians = []

    for i in range(5000):

        sample = np.random.choice(
            price_data,
            size=len(price_data),
            replace=True
        )

        bootstrap_medians.append(np.median(sample))

    lower = np.percentile(bootstrap_medians, 2.5)
    upper = np.percentile(bootstrap_medians, 97.5)

    results.append({
        "Class": label,
        "Median Price": np.median(price_data),
        "95% Bootstrap CI Lower": lower,
        "95% Bootstrap CI Upper": upper
    })

results = pd.DataFrame(results)

display(results.round(2))

,Class,Median Price,95% Bootstrap CI Lower,95% Bootstrap CI Upper
0,A,"18,200,000","18,000,000","18,500,000"
1,B,"8,000,000","8,000,000","8,100,000"
2,C,"3,350,000","3,300,000","3,400,000"
3,D,"1,350,000","1,325,000","1,360,000"


In [ ]:
# Test 4: Two-Sample Welch's t-test and Cohen's d comparing Apartment vs. Villa prices
results = []

for label, data in class_dfs.items():

    apartment_price = data[
        data["type"].str.lower() == "apartment"
    ]["price"].dropna()

    villa_price = data[
        data["type"].str.lower() == "villa"
    ]["price"].dropna()

    t_stat, p_value = stats.ttest_ind(
        apartment_price,
        villa_price,
        equal_var=False
    )

    n1 = len(apartment_price)
    n2 = len(villa_price)

    pooled_std = np.sqrt(
        (
            (n1 - 1) * apartment_price.var()
            + (n2 - 1) * villa_price.var()
        )
        / (n1 + n2 - 2)
    )

    cohens_d = (
        apartment_price.mean() - villa_price.mean()
    ) / pooled_std

    results.append({
        "Class": label,
        "Apartment Mean": apartment_price.mean(),
        "Villa Mean": villa_price.mean(),
        "t-statistic": t_stat,
        "p-value": p_value,
        "Cohen's d": cohens_d
    })

results = pd.DataFrame(results)

display(results.round(4))

,Class,Apartment Mean,Villa Mean,t-statistic,p-value,Cohen's d
0,A,"14,579,513","52,638,891",-34,0,-1
1,B,"7,207,748","22,702,745",-38,0,-3
2,C,"3,271,670","7,855,248",-15,0,-3
3,D,"1,378,522","3,876,839",-9,0,-4


In [ ]:
# Test 5: One-Way ANOVA (F-test) evaluating price differences across property types
results = []

for label, data in class_dfs.items():

    groups = []

    for property_type, group in data.groupby("type"):

        prices = group["price"].dropna()

        if len(prices) >= 30:
            groups.append(prices)

    if len(groups) >= 2:

        statistic, p_value = stats.f_oneway(*groups)

        results.append({
            "Class": label,
            "Number of Types": len(groups),
            "F-statistic": statistic,
            "p-value": p_value
        })

results = pd.DataFrame(results)

display(results.round(4))

,Class,Number of Types,F-statistic,p-value
0,A,8,418,0
1,B,8,"1,086",0
2,C,9,424,0
3,D,8,135,0


In [ ]:
# Test 6: Kruskal-Wallis Non-Parametric H-test comparing property type price distributions
results = []

for label, data in class_dfs.items():

    groups = []

    for property_type, group in data.groupby("type"):

        prices = group["price"].dropna()

        if len(prices) >= 30:
            groups.append(prices)

    if len(groups) >= 2:

        statistic, p_value = stats.kruskal(*groups)

        results.append({
            "Class": label,
            "H-statistic": statistic,
            "p-value": p_value
        })

results = pd.DataFrame(results)

display(results.round(4))

,Class,H-statistic,p-value
0,A,"4,508",0
1,B,"3,550",0
2,C,"2,120",0
3,D,257,0


In [ ]:
# Test 7: Spearman Rank Correlation between property size (sqm) and listing price
results = []

for label, data in class_dfs.items():

    analysis_data = data[
        ["size_sqm", "price"]
    ].dropna()

    rho, p_value = stats.spearmanr(
        analysis_data["size_sqm"],
        analysis_data["price"]
    )

    results.append({
        "Class": label,
        "Spearman ρ": rho,
        "p-value": p_value
    })

results = pd.DataFrame(results)

display(results.round(4))

,Class,Spearman ρ,p-value
0,A,1,0
1,B,1,0
2,C,1,0
3,D,0,0


In [ ]:
# Test 8: Chi-Square Test of Independence between property type and furnishing status
results = []

for label, data in class_dfs.items():

    contingency_table = pd.crosstab(
        data["type"],
        data["Furnished"]
    )

    chi2, p_value, dof, expected = stats.chi2_contingency(
        contingency_table
    )

    results.append({
        "Class": label,
        "Chi-square": chi2,
        "df": dof,
        "p-value": p_value
    })

results = pd.DataFrame(results)

display(results.round(4))

,Class,Chi-square,df,p-value
0,A,16,8,0
1,B,13,10,0
2,C,75,9,0
3,D,209,9,0


In [ ]:
# Test 9: Two-Sample Welch's t-test comparing Furnished vs. Unfurnished property prices
results = []

for label, data in class_dfs.items():

    unfurnished = data[
        data["Furnished"].str.lower() == "no"
    ]["price"].dropna()

    furnished = data[
        data["Furnished"].str.lower() == "yes"
    ]["price"].dropna()

    t_stat, p_value = stats.ttest_ind(
        unfurnished,
        furnished,
        equal_var=False
    )

    results.append({
        "Class": label,
        "Unfurnished Mean": unfurnished.mean(),
        "Furnished Mean": furnished.mean(),
        "t-statistic": t_stat,
        "p-value": p_value
    })

results = pd.DataFrame(results)

display(results.round(2))

,Class,Unfurnished Mean,Furnished Mean,t-statistic,p-value
0,A,"26,803,592","25,793,080",1,0
1,B,"10,183,813","9,859,535",1,0
2,C,"3,595,784","3,985,955",-3,0
3,D,"1,476,267","1,562,024",-1,0


In [ ]:
# Test 10: Spearman Rank Correlation Matrix across all numeric features per class
numeric_columns = [
    "price",
    "price_per_sqm",
    "size_sqm",
    "bedrooms",
    "bathrooms",
    "floor"
]

for label, data in class_dfs.items():

    correlation_matrix = (
        data[numeric_columns]
        .corr(method="spearman")
    )

    print(f"\n{'=' * 60}")
    print(f"CLASS {label}")
    print(f"{'=' * 60}")

    display(correlation_matrix.round(3))


CLASS A


,price,price_per_sqm,size_sqm,bedrooms,bathrooms,floor
price,1,0,1,1,1,NaN
price_per_sqm,0,1,-0,0,0,NaN
size_sqm,1,-0,1,1,1,NaN
bedrooms,1,0,1,1,1,NaN
bathrooms,1,0,1,1,1,NaN
floor,NaN,NaN,NaN,NaN,NaN,NaN



CLASS B


,price,price_per_sqm,size_sqm,bedrooms,bathrooms,floor
price,1,1,1,1,1,-0
price_per_sqm,1,1,0,0,0,0
size_sqm,1,0,1,1,1,-0
bedrooms,1,0,1,1,1,0
bathrooms,1,0,1,1,1,-0
floor,-0,0,-0,0,-0,1



CLASS C


,price,price_per_sqm,size_sqm,bedrooms,bathrooms,floor
price,1,0,1,1,1,0
price_per_sqm,0,1,-0,-0,0,-0
size_sqm,1,-0,1,1,1,0
bedrooms,1,-0,1,1,1,0
bathrooms,1,0,1,1,1,0
floor,0,-0,0,0,0,1



CLASS D


,price,price_per_sqm,size_sqm,bedrooms,bathrooms,floor
price,1,1,0,0,0,-0
price_per_sqm,1,1,-0,-0,0,-0
size_sqm,0,-0,1,1,1,-0
bedrooms,0,-0,1,1,1,0
bathrooms,0,0,1,1,1,-0
floor,-0,-0,-0,0,-0,1
